In [35]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import MultiLabelBinarizer
import torch
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim


In [2]:
df=pd.read_csv("D:\\sarah_venv\\archive (5)\\mLabel_tweets.csv")
df.head()

,ID,tweet,labels
0,1296010336907038720t,@cath__kath AstraZeneca is made with the kidne...,ingredients
1,1336808189677940736t,It begins. Please find safe alternatives to th...,side-effect
2,1329488407307956231t,"@PaolaQP1231 Well, I mean congratulations Covi...",side-effect
3,1364194604459900934t,@BorisJohnson for those of us that do not wish...,mandatory
4,1375938799247765515t,She has been trying to speak out: writing lett...,side-effect rushed


In [3]:
df['tweet']=df['tweet'].apply(lambda x:x.split())

In [4]:


mlb = MultiLabelBinarizer(sparse_output=True)

y_train = mlb.fit_transform(df['tweet'])
y_test  = mlb.transform(df['labels'])
num_classes=len(mlb.classes_)

d:\sarah_venv\sarah_venv\Lib\site-packages\sklearn\preprocessing\_label.py:909: UserWarning: unknown class(es) [' ', 'c', 'h', 'p'] will be ignored
  warnings.warn(


In [5]:
emb={}
with open("D:\\sarah_venv\\glove100d\\glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values=line.split()
        word=values[0]
        coef=np.asarray(values[1:],dtype='float32')
        emb[word]=coef
    print("loaded %d word vectors", len(emb))

loaded %d word vectors 400000


In [6]:
def text_to_glove_vector(text):
    words=[w for w in text.lower().split() if w not in ENGLISH_STOP_WORDS]
    vectors=[emb[w] for w in words if w in emb]
    if len(vectors)==0:
        return np.zeros(100)
    return np.mean(vectors,axis=0)

x_ttrain = np.array([text_to_glove_vector(t) for t in tqdm(str(df['tweet']))])

x_test=np.array([text_to_glove_vector(t) for t in tqdm(str(df['labels']))])

100%|██████████| 678/678 [00:00<00:00, 115636.72it/s]


100%|██████████| 338/338 [00:00<00:00, 92314.56it/s]


In [27]:
class ReuterDataset(Dataset):
    def __init__(self, x, y):
        # x is dense numpy → OK
        self.x = torch.tensor(x, dtype=torch.float32)

        # y is sparse CSR → keep it sparse
        self.y = y

    def __len__(self):
        return self.x.shape[0]   # always valid

    def __getitem__(self, idx):
        # y[idx] is 1 × num_labels sparse row → convert just 1 row
        y_row = torch.tensor(self.y[idx].ravel(),
                             dtype=torch.float32)
        return self.x[idx], y_row


In [29]:
train_dataset = ReuterDataset(x_ttrain, y_train)
test_dataset  = ReuterDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=8)


In [30]:
class MLPClassifier(nn.Module):
    def __init__(self, ip_dim, op_dim):
        super(MLPClassifier,self).__init__()
        self.fc1=nn.Linear(ip_dim,128)
        self.relu=nn.ReLU()
        self.fc2=nn.Linear(128,op_dim)
        self.sigmoid=nn.Sigmoid()

    def forward (self,x):
        x=self.relu(self.fc1(x))
        x=self.sigmoid(self.fc2(x))
        return x
    
model=MLPClassifier(ip_dim=100,op_dim=num_classes)

    

In [31]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [36]:
epochs=5
for epoch in range(epochs):
    model.train()
    total_loss=0
for x_batch,y_batch in train_loader:
    optimizer.zero_grad()
    outputs=model(x_batch)
    loss=criterion(outputs,y_batch)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    

print(float(total_loss)/len(train_loader))


0.10952008162241648
